# Heat Risk Index — Sensitivity Analysis & VCP Comparison

Three analyses:

1. **Weighting sensitivity** — compare facility rankings under three component-weighting schemes (current/historic period only)
2. **Own-tract VCP validation** — correlate our risk score against VCP's `ExHeatHealth_Idx` for each prison's host census tract
3. **Surrounding community comparison** — adjacent non-institutional tract VCP heat values vs. our index, as a methodological argument for a prison-specific index

Output: `data/cdcr/CDCR_heat_risk_sensitivity.csv`

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
from scipy import stats

# Load risk index — mid-century only
risk = pd.read_csv('data/cdcr/CDCR_heat_risk_index_additive_25_25_50.csv')
mc = risk[risk['time_period'] == 'current'].copy().reset_index(drop=True)
print(f'Facilities: {len(mc)}')
print(mc[['cdcr_code', 'hazard_score', 'exposure_score', 'vulnerability_score', 'risk_score']].to_string(index=False))

# Load facilities for tract_geoid join
fac = pd.read_csv('data/cdcr/cdcr_facilities.csv')[['cdcr_code', 'tract_geoid']]
mc = mc.merge(fac, on='cdcr_code', how='left')
mc['tract_str'] = mc['tract_geoid'].astype(str).str.split('.').str[0].str.zfill(11)
print(f'\ntract_geoid nulls: {mc["tract_geoid"].isnull().sum()}')

## 1. Weighting Sensitivity

Three schemes tested against additive vulnerability-upweighted (current index):

| Scheme | Formula | Logic |
|---|---|---|
| A — Additive 25/25/50 (current) | 0.25×H + 0.25×E + 0.50×V, normalized 0–100 | Ovienmhada vulnerability upweighting; additive — high vulnerability alone can drive rank |
| B — Equal multiplicative | H × E × V, normalized 0–100 | All components equal; risk requires elevation in all three |
| C — Multiplicative V² | H × E × V², normalized 0–100 | Preserves multiplicative structure; vulnerability amplified but still requires hazard and exposure |

Scores for all three are normalized to 0–100 within mid-century facilities only.

In [ ]:
H = mc['hazard_score']
E = mc['exposure_score']
V = mc['vulnerability_score']

def norm100(s):
    mn, mx = s.min(), s.max()
    return (s - mn) / (mx - mn) * 100

# Scheme A: additive 25/25/50 (current index default)
mc['score_A'] = norm100(0.25 * H + 0.25 * E + 0.50 * V)

# Scheme B: equal multiplicative
mc['score_B'] = norm100(H * E * V)

# Scheme C: multiplicative V²
mc['score_C'] = norm100(H * E * V**2)

# Ranks (1 = highest risk)
for col, rank_col in [('score_A', 'rank_A'), ('score_B', 'rank_B'), ('score_C', 'rank_C')]:
    mc[rank_col] = mc[col].rank(ascending=False).astype(int)

# Spearman correlations between schemes
pairs = [('A', 'B'), ('A', 'C'), ('B', 'C')]
print('Spearman rank correlations (mid-century):')
for s1, s2 in pairs:
    r, p = stats.spearmanr(mc[f'rank_{s1}'], mc[f'rank_{s2}'])
    print(f'  {s1} vs {s2}: r={r:.3f}, p={p:.4f}')

print()

# Full rank comparison table
rank_cols = ['cdcr_code', 'rank_A', 'rank_B', 'rank_C']
rank_df = mc[rank_cols].copy()
rank_df['max_swing'] = rank_df[['rank_A','rank_B','rank_C']].max(axis=1) - rank_df[['rank_A','rank_B','rank_C']].min(axis=1)
rank_df = rank_df.sort_values('rank_A')
print('Rankings by scheme (sorted by Scheme A):')
print(rank_df.to_string(index=False))

In [ ]:
# Facilities with largest rank swings across all three schemes
print('Facilities with max rank swing ≥ 5 positions:')
large_swings = rank_df[rank_df['max_swing'] >= 5].sort_values('max_swing', ascending=False)
if len(large_swings) == 0:
    print('  None — all facilities stable within 4 rank positions across schemes')
else:
    print(large_swings.to_string(index=False))

print()
print('Top 10 by Scheme A (current equal-weight):')
print(mc[['cdcr_code','name','score_A','score_B','score_C','rank_A','rank_B','rank_C']]
      .sort_values('rank_A').head(10).round(1).to_string(index=False))

## 2. Own-Tract VCP Comparison

Correlates our mid-century risk score against VCP's `ExHeatHealth_Idx` for each prison's host census tract.

`ExHeatHealth_Idx` is VCP's combined extreme heat health index — a composite of heat hazard (days over 100°F, hot nights) and social vulnerability (chronic disease, income, age, disability). It is available for all census tracts including prison tracts.

**Expected finding:** Low correlation. VCP's social vulnerability component uses community-level demographics (income, renters, limited English households) that do not describe the incarcerated population. This divergence is the methodological argument for a prison-specific index rather than a limitation of ours.

In [ ]:
# Load VCP
vcp = gpd.read_file('data_sources/hazards/VCP_Tracts.geojson')
vcp['GEOID_str'] = vcp['GEOID'].astype(str)

vcp_cols = ['GEOID_str', 'Pct_GroupQuarters', 'ExHeatHealth_Idx',
            'Heat_sc_hazard_pre', 'Heat_sc_hazard_fut', 'Heat_sc_social',
            'INDEX_PCTL', 'geometry']
vcp_slim = vcp[vcp_cols].copy()

# Join prison tracts
mc_vcp = mc.merge(
    vcp_slim.drop(columns='geometry'),
    left_on='tract_str', right_on='GEOID_str', how='left'
)

n_matched = mc_vcp['ExHeatHealth_Idx'].notna().sum()
print(f'Facilities matched to VCP tract: {n_matched} of {len(mc_vcp)}')
print(f'Pct_GroupQuarters for prison tracts:')
print(mc_vcp[['cdcr_code', 'tract_str', 'Pct_GroupQuarters', 'ExHeatHealth_Idx', 'INDEX_PCTL']]
      .sort_values('Pct_GroupQuarters', ascending=False).to_string(index=False))

print()
# Pearson and Spearman vs our risk score
pair = mc_vcp[['risk_score', 'ExHeatHealth_Idx']].dropna()
r_p, p_p = stats.pearsonr(pair['risk_score'], pair['ExHeatHealth_Idx'])
r_s, p_s = stats.spearmanr(pair['risk_score'], pair['ExHeatHealth_Idx'])
print(f'Our risk_score vs VCP ExHeatHealth_Idx (n={len(pair)}):')
print(f'  Pearson  r={r_p:.3f}, p={p_p:.4f}')
print(f'  Spearman r={r_s:.3f}, p={p_s:.4f}')

# Also compare just hazard components where available
pair_h = mc_vcp[['hazard_score', 'Heat_sc_hazard_fut']].dropna()
r_h, p_h = stats.spearmanr(pair_h['hazard_score'], pair_h['Heat_sc_hazard_fut'])
print(f'\nOur hazard_score vs VCP Heat_sc_hazard_fut (n={len(pair_h)}):')
print(f'  Spearman r={r_h:.3f}, p={p_h:.4f}')

## 3. Surrounding Community Comparison

For each prison, find adjacent VCP census tracts (sharing a border), exclude tracts with `Pct_GroupQuarters > 25%` (other institutional tracts), and average their `ExHeatHealth_Idx`.

This answers: *How does VCP assess heat risk for the communities surrounding each prison?* The comparison between that community-facing index and our prison-specific index illustrates why a dedicated framework is needed — VCP's community variables (income, renters, chronic disease prevalence in free populations) do not translate to the incarcerated context.

In [ ]:
# Build spatial adjacency: for each prison tract, find touching VCP tracts
# vcp is already a GeoDataFrame with geometry

# Ensure consistent CRS
vcp_geo = vcp[['GEOID_str', 'Pct_GroupQuarters', 'ExHeatHealth_Idx', 'geometry']].copy()
vcp_geo = vcp_geo.set_index('GEOID_str')

community_rows = []

for _, row in mc_vcp.iterrows():
    tract = row['tract_str']
    if tract not in vcp_geo.index:
        community_rows.append({'cdcr_code': row['cdcr_code'], 'n_neighbors': 0,
                                'community_ExHeatHealth_Idx': np.nan})
        continue

    prison_geom = vcp_geo.loc[tract, 'geometry']

    # Find tracts that touch this one (shared border, not just point)
    neighbors = vcp_geo[
        (vcp_geo.index != tract) &
        (vcp_geo['geometry'].touches(prison_geom) | vcp_geo['geometry'].intersects(prison_geom)) &
        (vcp_geo.index != tract)
    ].copy()

    # Exclude other institutional tracts
    community = neighbors[neighbors['Pct_GroupQuarters'] <= 25]

    n = len(community)
    avg_idx = community['ExHeatHealth_Idx'].mean() if n > 0 else np.nan

    community_rows.append({
        'cdcr_code': row['cdcr_code'],
        'n_neighbors': n,
        'community_ExHeatHealth_Idx': avg_idx
    })

community_df = pd.DataFrame(community_rows)
mc_vcp = mc_vcp.merge(community_df, on='cdcr_code', how='left')

print('Community adjacent tract count and avg ExHeatHealth_Idx:')
print(mc_vcp[['cdcr_code', 'n_neighbors', 'community_ExHeatHealth_Idx', 'ExHeatHealth_Idx']]
      .sort_values('community_ExHeatHealth_Idx', ascending=False).to_string(index=False))

In [ ]:
# Summary comparison: our risk rank vs community VCP rank
mc_vcp['community_rank'] = mc_vcp['community_ExHeatHealth_Idx'].rank(ascending=False)

print('Prison risk rank (our index) vs surrounding community heat rank (VCP):')
compare = mc_vcp[['cdcr_code', 'name', 'risk_score', 'rank_A',
                   'community_ExHeatHealth_Idx', 'community_rank', 'n_neighbors']].copy()
compare['rank_diff'] = (compare['rank_A'] - compare['community_rank']).round(0).astype('Int64')
compare = compare.sort_values('rank_A')
print(compare.to_string(index=False))

print()
# Correlation between our risk rank and community VCP rank
pair_c = compare[['risk_score', 'community_ExHeatHealth_Idx']].dropna()
r_c, p_c = stats.spearmanr(pair_c['risk_score'], pair_c['community_ExHeatHealth_Idx'])
print(f'Spearman r (our risk_score vs community ExHeatHealth_Idx): r={r_c:.3f}, p={p_c:.4f} (n={len(pair_c)})')
print()
print('Positive rank_diff = our index ranks facility HIGHER risk than VCP ranks surrounding community')
print('Negative rank_diff = VCP ranks surrounding community higher than our index ranks the prison')

## 4. Output CSV

Saves per-facility scores and ranks under all three weighting schemes, plus VCP comparison values, to `data/cdcr/CDCR_heat_risk_sensitivity.csv`.

In [ ]:
# Load pct_units_refrigeration from facilities
fac_ac = pd.read_csv('data/cdcr/cdcr_facilities.csv')[['cdcr_code', 'pct_units_refrigeration']]
mc_vcp = mc_vcp.merge(fac_ac, on='cdcr_code', how='left')

out_cols = [
    'cdcr_code', 'name', 'average_2025_population', 'pct_units_refrigeration',
    'hazard_score', 'exposure_score', 'vulnerability_score',
    # Weighting sensitivity
    'score_A', 'score_B', 'score_C',
    'rank_A', 'rank_B', 'rank_C',
    # VCP own-tract
    'tract_str', 'Pct_GroupQuarters', 'ExHeatHealth_Idx',
    'Heat_sc_hazard_fut', 'INDEX_PCTL',
    # VCP surrounding community
    'n_neighbors', 'community_ExHeatHealth_Idx',
]

output = mc_vcp[out_cols].copy()
output = output.rename(columns={
    'tract_str': 'tract_geoid',
    'score_A': 'risk_score_additive_25_25_50',
    'score_B': 'risk_score_equal_mult',
    'score_C': 'risk_score_mult_vsq',
    'rank_A':  'rank_additive_25_25_50',
    'rank_B':  'rank_equal_mult',
    'rank_C':  'rank_mult_vsq',
    'ExHeatHealth_Idx': 'vcp_own_tract_ExHeatHealth_Idx',
    'Heat_sc_hazard_fut': 'vcp_own_tract_heat_hazard_fut',
    'INDEX_PCTL': 'vcp_own_tract_INDEX_PCTL',
    'community_ExHeatHealth_Idx': 'vcp_community_ExHeatHealth_Idx',
})

score_cols = ['risk_score_additive_25_25_50', 'risk_score_equal_mult', 'risk_score_mult_vsq',
              'vcp_own_tract_ExHeatHealth_Idx', 'vcp_own_tract_heat_hazard_fut',
              'vcp_own_tract_INDEX_PCTL', 'vcp_community_ExHeatHealth_Idx']
output[score_cols] = output[score_cols].round(2)

output.to_csv('data/cdcr/CDCR_heat_risk_sensitivity.csv', index=False)
print(f'Saved {len(output)} rows to data/cdcr/CDCR_heat_risk_sensitivity.csv')
print(f'Columns: {list(output.columns)}')